# Code-Capacity Half-Stabilizer Gaussian Study

This notebook focuses on half-stabilizer behaviour, including Gaussian refinement, practical K-draw aggregation, and equivalence reporting between deterministic and rerolled strategies.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from relay_bp.analysis import (
    EquivalenceThresholds,
    MemorySamplingSpec,
    bootstrap_confidence_interval,
    default_export_code_paths,
    enumerate_half_stabilizer_cases,
    evaluate_equivalence,
    evaluate_gaussian_refinement_heatmap,
    evaluate_sampling_strategy,
    gaussian_refinement_heatmap_rows,
    load_code_capacity_problem,
    with_uniform_error_rate,
)

CODE_PATHS = default_export_code_paths(REPO_ROOT)
selected_code = "surface13"
p_random_error = 0.09


c:\Users\User\Documents\projects-git\relay_mother_folder\relay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
base_problem = load_code_capacity_problem(CODE_PATHS[selected_code])
problem = with_uniform_error_rate(base_problem, p_random_error)
half_cases = enumerate_half_stabilizer_cases(problem, max_cases=64, shuffle_seed=0)
refinement = evaluate_gaussian_refinement_heatmap(
    problem=problem,
    cases=half_cases,
    max_iter=20,
    alpha=1.0,
    means=np.linspace(-0.1, 0.3, 5).tolist(),
    sigmas=np.linspace(0.01, 0.15, 5).tolist(),
    low=-0.1,
    high=0.3,
    base_seed=17,
    random_draws_per_point=2,
    application_scope="stabilizer_support",
    selection_mode="single_draw",
    logical_weight_penalty=4.0,
    convergence_penalty=1.0,
    iteration_penalty=0.05,
)
refinement_rows = pd.DataFrame(gaussian_refinement_heatmap_rows(refinement))
display(refinement_rows.sort_values(["loss_mean", "mean_iterations"]).head(8))


,mean,sigma,low,high,logical_success_rate,convergence_rate,exact_recovery_rate,mean_iterations,mean_logical_weight,loss_mean,trial_count
24,0.3,0.150,-0.1,0.3,1.0,1.000000,0.289062,4.851562,0.0,0.012129,64
14,0.3,0.080,-0.1,0.3,1.0,1.000000,0.328125,4.875000,0.0,0.012188,64
23,0.2,0.150,-0.1,0.3,1.0,1.000000,0.351562,5.242188,0.0,0.013105,64
9,0.3,0.045,-0.1,0.3,1.0,1.000000,0.320312,5.265625,0.0,0.013164,64
8,0.2,0.045,-0.1,0.3,1.0,0.992188,0.273438,4.921875,0.0,0.020117,64
18,0.2,0.115,-0.1,0.3,1.0,0.992188,0.226562,5.195312,0.0,0.020801,64
13,0.2,0.080,-0.1,0.3,1.0,0.992188,0.312500,5.250000,0.0,0.020937,64
19,0.3,0.115,-0.1,0.3,1.0,0.984375,0.343750,5.375000,0.0,0.029063,64


In [4]:
practical_spec = MemorySamplingSpec(
    sampling_family="truncated_gaussian",
    application_scope="stabilizer_support",
    selection_mode="k_draw_practical",
    draw_count=4,
    low=-0.1,
    high=0.3,
    mean=0.1,
    sigma=0.05,
)
practical_metrics = evaluate_sampling_strategy(
    problem=problem,
    cases=half_cases,
    sampling_spec=practical_spec,
    max_iter=20,
    alpha=1.0,
    logical_weight_penalty=4.0,
    convergence_penalty=1.0,
    iteration_penalty=0.05,
    base_seed=5,
)
pd.DataFrame([practical_metrics])


,num_cases,loss_mean,logical_success_rate,exact_recovery_rate,convergence_rate,mean_iterations,mean_logical_weight
0,64,0.016406,1.0,0.3125,0.984375,0.3125,0.0


In [5]:
equivalence = evaluate_equivalence(
    candidate_rows=[
        {"logical_success_rate": 0.80, "convergence_rate": 0.70, "mean_iterations": 8.8},
        {"logical_success_rate": 0.82, "convergence_rate": 0.73, "mean_iterations": 8.5},
    ],
    baseline_rows=[
        {"logical_success_rate": 0.81, "convergence_rate": 0.72, "mean_iterations": 8.7},
        {"logical_success_rate": 0.82, "convergence_rate": 0.73, "mean_iterations": 8.6},
    ],
    thresholds=EquivalenceThresholds(bootstrap_samples=512, seed=3),
)
pd.DataFrame([equivalence["logical_success_difference"], equivalence["convergence_difference"]])


,mean,lower,upper
0,-0.005,-0.01,0.0
1,-0.010,-0.02,0.0


In [6]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root.")


_repo_root = find_repo_root(Path.cwd())
_src_root = _repo_root / "src"
if str(_src_root) not in sys.path:
    sys.path.insert(0, str(_src_root))

from relay_bp.analysis import (
    MemorySamplingSpec,
    bootstrap_confidence_interval,
    default_export_code_paths,
    enumerate_half_stabilizer_cases,
    evaluate_sampling_strategy,
    load_code_capacity_problem,
    with_uniform_error_rate,
)

_code_paths = default_export_code_paths(_repo_root)
smoke_problem = with_uniform_error_rate(load_code_capacity_problem(_code_paths["surface13"]), 0.09)
smoke_cases = enumerate_half_stabilizer_cases(smoke_problem, max_cases=8, shuffle_seed=2)
smoke_spec = MemorySamplingSpec(
    sampling_family="uniform_interval",
    application_scope="stabilizer_support",
    selection_mode="k_draw_practical",
    draw_count=2,
    low=0.0,
    high=0.2,
)
smoke_metrics = evaluate_sampling_strategy(
    problem=smoke_problem,
    cases=smoke_cases,
    sampling_spec=smoke_spec,
    max_iter=10,
    alpha=1.0,
    logical_weight_penalty=4.0,
    convergence_penalty=1.0,
    iteration_penalty=0.05,
    base_seed=13,
)
assert smoke_metrics["num_cases"] == len(smoke_cases)
assert "mean" in bootstrap_confidence_interval([0.0, 1.0, 1.0], bootstrap_samples=64, seed=1)
